In [4]:
import pandas as pd
import torch
import torch.nn as nn
from torch.nn import functional as F

In [5]:
with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [6]:
chars = sorted(list(set(text))) # ambil karakter yang ada di dataset
vocab_size = len(chars) # panjang karakter yang ada di dataset

for i in range(len(chars)):
    print(f"{i} : {chars[i]}")
print(f'vocab_size: {vocab_size}')

0 : 

1 :  
2 : !
3 : $
4 : &
5 : '
6 : ,
7 : -
8 : .
9 : 3
10 : :
11 : ;
12 : ?
13 : A
14 : B
15 : C
16 : D
17 : E
18 : F
19 : G
20 : H
21 : I
22 : J
23 : K
24 : L
25 : M
26 : N
27 : O
28 : P
29 : Q
30 : R
31 : S
32 : T
33 : U
34 : V
35 : W
36 : X
37 : Y
38 : Z
39 : a
40 : b
41 : c
42 : d
43 : e
44 : f
45 : g
46 : h
47 : i
48 : j
49 : k
50 : l
51 : m
52 : n
53 : o
54 : p
55 : q
56 : r
57 : s
58 : t
59 : u
60 : v
61 : w
62 : x
63 : y
64 : z
vocab_size: 65


In [7]:
stoi = {ch: i for i,ch in enumerate(chars)} # mapping character ke integers
itos = {i: ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s] # mengubah string menjadi integers (atau memiliki id)
decode = lambda l: ''.join([itos[i] for i in l]) # mengubah id atau integers menjadi string

# contoh 
nama= "Ahmad Mufli Ramadhan"
print(f"Setelah di encode: {encode(nama)}") # diubah dari string menjadi integers
print(f"Setelah di decode: {decode(encode(nama))}") # diubah dari integers ke string
print(decode([25, 59, 44, 50, 47]))

Setelah di encode: [13, 46, 51, 39, 42, 1, 25, 59, 44, 50, 47, 1, 30, 39, 51, 39, 42, 46, 39, 52]
Setelah di decode: Ahmad Mufli Ramadhan
Mufli


In [8]:
data = torch.tensor(encode(text), dtype=torch.long) # konversi data dari text ke tensor

In [9]:
# Split data 80/20
n = int(0.8*len(data))
train_data = data[:n] # 0 sampai n
val_data = data[n:] # n -> size of the dataset
print(train_data)

tensor([18, 47, 56,  ..., 39, 58,  1])


In [10]:
context_length = 8 # Panjang chunk atau konteks yang kita inginkan untuk dilihat oleh model
X = train_data[:context_length] # selalu mulai dari 0 -> context length
y = train_data[1:context_length+1] # selalu mulai dari 1 -> context length + 1

for i in range(context_length):
    print(f"Ketika input adalah {X[:i+1]} maka yang diprediksi adalah {y[i]}")

Ketika input adalah tensor([18]) maka yang diprediksi adalah 47
Ketika input adalah tensor([18, 47]) maka yang diprediksi adalah 56
Ketika input adalah tensor([18, 47, 56]) maka yang diprediksi adalah 57
Ketika input adalah tensor([18, 47, 56, 57]) maka yang diprediksi adalah 58
Ketika input adalah tensor([18, 47, 56, 57, 58]) maka yang diprediksi adalah 1
Ketika input adalah tensor([18, 47, 56, 57, 58,  1]) maka yang diprediksi adalah 15
Ketika input adalah tensor([18, 47, 56, 57, 58,  1, 15]) maka yang diprediksi adalah 47
Ketika input adalah tensor([18, 47, 56, 57, 58,  1, 15, 47]) maka yang diprediksi adalah 58


In [11]:
torch.manual_seed(1337)
batch_size = 4 # berapa banyak independent sequences yang akan di proses secara pararel

def get_batch(split):
    # membuat batch kecil dari data dari input X dan target Y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_length, (batch_size,)) # mengambil posisi kalimat secara acak yang ada di dalam corpus dataset kita
    x = torch.stack([data[i:i+context_length] for i in ix]) # kemudian setiap kalimat yang posisinya sudah di ambil, akan di stack secara bersamaan
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    return x, y

ix = torch.randint((len(train_data) - context_length), (batch_size,)) # ini mengambil posisi kalimat yang ada di dalam corpus kita atau dataset
print(f"ix : {ix}")
for i in ix:
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:i+context_length] for i in ix])
    print(x)
    print(f" round {i}: {data[i:i+context_length]}")

ix : tensor([188288,  12671, 712251, 226939])


tensor([[58, 63,  8,  0,  0, 19, 24, 27],
        [39, 59, 45, 46, 58,  1, 46, 43],
        [49, 43, 57,  1, 53, 50, 42,  1],
        [52, 41, 47, 43, 52, 58,  1, 56]])
 round 188288: tensor([58, 63,  8,  0,  0, 19, 24, 27])
tensor([[58, 63,  8,  0,  0, 19, 24, 27],
        [39, 59, 45, 46, 58,  1, 46, 43],
        [49, 43, 57,  1, 53, 50, 42,  1],
        [52, 41, 47, 43, 52, 58,  1, 56]])
 round 12671: tensor([39, 59, 45, 46, 58,  1, 46, 43])
tensor([[58, 63,  8,  0,  0, 19, 24, 27],
        [39, 59, 45, 46, 58,  1, 46, 43],
        [49, 43, 57,  1, 53, 50, 42,  1],
        [52, 41, 47, 43, 52, 58,  1, 56]])
 round 712251: tensor([49, 43, 57,  1, 53, 50, 42,  1])
tensor([[58, 63,  8,  0,  0, 19, 24, 27],
        [39, 59, 45, 46, 58,  1, 46, 43],
        [49, 43, 57,  1, 53, 50, 42,  1],
        [52, 41, 47, 43, 52, 58,  1, 56]])
 round 226939: tensor([52, 41, 47, 43, 52, 58,  1, 56])


In [12]:
xb, yb = get_batch('train')
print("Inputs: ")
print(xb.shape)
print(xb)

print('targets: ')
print(yb.shape)
print(yb)

Inputs: 
torch.Size([4, 8])
tensor([[43, 56, 63,  1, 47, 52,  1, 46],
        [44, 47, 45, 46, 58,  1, 40, 43],
        [58, 46, 43, 56,  8,  0,  0, 24],
        [ 6,  0, 21,  1, 52, 43, 60, 43]])
targets: 
torch.Size([4, 8])
tensor([[56, 63,  1, 47, 52,  1, 46, 47],
        [47, 45, 46, 58,  1, 40, 43,  1],
        [46, 43, 56,  8,  0,  0, 24, 17],
        [ 0, 21,  1, 52, 43, 60, 43, 56]])


In [13]:
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # setiap token langsung membaca dari logits untuk token berikutnya dari lookup table
    
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B, T, C) Batch, Time (Vocab_size), Channel
        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # Menyesuaikan dengan argument yang dibutuhkan oleh cross entropy yaitu 2d vector
            targets = targets.view(-1) # mengubah menjadi 1d vector atau B*T
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx adalah Batch, Time (B, T) indeks array dari konteks sekarang 
        for _ in range(max_new_tokens):
            # Mengambil prediksi
            logits, _ = self(idx)
            # fokus hanya untuk step terakhir 
            logits = logits[:, -1, :] # berubah menjadi (B, C) dari (B, T)
            # Gunakan activation function softmax untuk mendapatkan probabilitas
            probs = F.softmax(logits, dim=-1) # Dim (B, C)
            # sample dari hasil softmax
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
    

m = BigramLanguageModel(vocab_size)
out, loss = m(xb, yb)
print(out.shape)
print(loss)

idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.5193, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [14]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) # optimizer 

In [15]:
# train the model
batch_size = 32
epochs = 10000

for steps in range(epochs):
    xb, yb = get_batch('train')

    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())  

2.5331473350524902


In [16]:
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))


lso br. ave aviu:


Smy, may be ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof wan!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty dedo bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
By bre ndy; by s afreanoo adicererupa anse tecorro llaus a!
OLengerithesththenou theal amas trr
TI ar I t, mes, o IUSt my w,

Whank'the
THek' merer, dd
We ntem lud engitonso; cer ize helour
Jginte the?
Thak orblyor id oree chot, p,
Bealivolde Th ll


#### Trik matematika di self attention

In [26]:
torch.manual_seed(1337)
B,T,C = 4, 8, 2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [41]:
x[0, 1, 0]

tensor(-0.3596)

In [22]:
xbow = torch.zeros((B,T,C))
xbow.shape

torch.Size([4, 8, 2])

In [23]:
xbow = torch.zeros((B,T,C))

for b in range(B):
    for t in range (T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev, 0)